In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PostgresConnector") \
    .config("spark.jars","/usr/lib/spark/jars/postgresql-42.7.4.jar")\
    .getOrCreate()


In [ ]:
# Question 31: Employees who earn more than their current manager.
# Concept: Complex Join
employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department_employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})
salary_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.salary",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})
department_manager_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department_manager",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})
# Solution:
mgr_cond = [department_manager_df.employee_id == salary_df.employee_id, salary_df.to_date == '9999-01-01',
            department_manager_df.to_date == '9999-01-01']
managers = department_manager_df.join(salary_df, mgr_cond) \
                               .select(department_manager_df.department_id, salary_df.amount.alias("mgr_amount"))

emp_cond = [employee_df.id == salary_df.employee_id, salary_df.to_date == '9999-01-01',
            employee_df.id == department_employee_df.employee_id, department_employee_df.to_date == '9999-01-01',
            department_employee_df.department_id == department_df.id]

emps = employee_df.join(salary_df, employee_df.id == salary_df.employee_id) \
                  .join(department_employee_df, employee_df.id == department_employee_df.employee_id) \
                  .join(department_df, department_employee_df.department_id == department_df.id)

emps.join(managers, emps.department_id == managers.department_id) \
    .filter(emps.amount > managers.mgr_amount) \
    .select(employee_df.id, department_df.dept_name, emps.amount) \
    .show()
